# Домашнее задание: Несбалансированные классы и Pipeline

### Цели работы
1. Понять проблему несбалансированных классов и почему accuracy не подходит
2. Реализовать метрики классификации: confusion matrix, precision, recall, F1
3. Реализовать методы работы с дисбалансом: oversampling, SMOTE, undersampling
4. Научиться использовать `class_weight` в sklearn моделях
5. Сравнить ROC-AUC и PR-AUC для оценки моделей
6. Освоить sklearn Pipeline для предотвращения утечки данных
7. Использовать ColumnTransformer для смешанных типов признаков

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (confusion_matrix as sk_confusion_matrix,
                             precision_score, recall_score, f1_score,
                             roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score,
                             accuracy_score, classification_report)
import ipytest
import pytest

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

## Теория: Несбалансированные классы

### Что такое дисбаланс классов?

В реальных задачах классы часто распределены неравномерно:
- Обнаружение мошенничества: ~0.1% мошеннических транзакций
- Медицинская диагностика: ~1% больных среди обследуемых
- Отток клиентов: ~5% уходят в месяц

### Почему accuracy не подходит?

Если 95% транзакций легитимные, модель, которая **всегда** предсказывает "легитимно",
получит accuracy = 95%, но не найдёт ни одного мошенничества!

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

### Confusion Matrix

|  | Предсказан + | Предсказан - |
|---|---|---|
| **Реальный +** | TP (True Positive) | FN (False Negative) |
| **Реальный -** | FP (False Positive) | TN (True Negative) |

### Precision, Recall, F1

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(какая доля предсказанных позитивов реально позитивны?)}$$

$$\text{Recall} = \frac{TP}{TP + FN} \quad \text{(какую долю реальных позитивов мы нашли?)}$$

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Trade-off:** Увеличивая порог классификации, мы повышаем precision (меньше ложных срабатываний), но снижаем recall (пропускаем больше позитивов).

### ROC и PR кривые

**ROC-кривая:** зависимость TPR от FPR при разных порогах.
- TPR (Recall) = TP / (TP + FN)
- FPR = FP / (FP + TN)
- ROC-AUC: вероятность, что случайный позитив ранжирован выше случайного негатива

**PR-кривая:** зависимость Precision от Recall. **Более информативна при дисбалансе!**

### Подходы к решению

| Метод | Описание |
|---|---|
| **Random Oversampling** | Дублирование объектов миноритарного класса |
| **SMOTE** | Генерация синтетических объектов миноритарного класса |
| **Random Undersampling** | Удаление объектов мажоритарного класса |
| **class_weight** | Увеличение штрафа за ошибку на миноритарном классе |

## Задание 1: Метрики классификации с нуля

Реализуйте функции для вычисления confusion matrix, precision, recall и F1.
Используйте только операции с массивами numpy.

In [ ]:
def my_confusion_matrix(y_true, y_pred):
    '''Compute confusion matrix for binary classification.
    Returns: dict with keys TP, FP, FN, TN.'''
    #TODO: Compute TP, FP, FN, TN using boolean indexing
    pass


def my_precision(y_true, y_pred):
    '''Precision = TP / (TP + FP). Return 0.0 if denominator is 0.'''
    #TODO: Implement
    pass


def my_recall(y_true, y_pred):
    '''Recall = TP / (TP + FN). Return 0.0 if denominator is 0.'''
    #TODO: Implement
    pass


def my_f1(y_true, y_pred):
    '''F1 = 2 * P * R / (P + R). Return 0.0 if both are 0.'''
    #TODO: Implement
    pass

In [ ]:
ipytest.autoconfig()


class TestMetrics:
    def test_confusion_matrix(self):
        y_true = np.array([1, 0, 1, 1, 0, 0, 1, 0])
        y_pred = np.array([1, 0, 0, 1, 0, 1, 1, 0])
        cm = my_confusion_matrix(y_true, y_pred)
        assert cm['TP'] == 3
        assert cm['FP'] == 1
        assert cm['FN'] == 1
        assert cm['TN'] == 3

    def test_precision(self):
        y_true = np.array([1, 0, 1, 1, 0])
        y_pred = np.array([1, 0, 0, 1, 1])
        assert np.isclose(my_precision(y_true, y_pred), 2 / 3)

    def test_recall(self):
        y_true = np.array([1, 0, 1, 1, 0])
        y_pred = np.array([1, 0, 0, 1, 1])
        assert np.isclose(my_recall(y_true, y_pred), 2 / 3)

    def test_f1(self):
        assert np.isclose(my_f1(np.array([1, 1, 0]), np.array([1, 0, 0])), 2 / 3)

    def test_edge_case_all_negative(self):
        y_true = np.array([0, 0, 0])
        y_pred = np.array([0, 0, 0])
        assert my_precision(y_true, y_pred) == 0.0
        assert my_recall(y_true, y_pred) == 0.0

    def test_against_sklearn(self):
        np.random.seed(RANDOM_STATE)
        y_true = np.random.randint(0, 2, 100)
        y_pred = np.random.randint(0, 2, 100)
        assert np.isclose(my_precision(y_true, y_pred), precision_score(y_true, y_pred, zero_division=0))
        assert np.isclose(my_recall(y_true, y_pred), recall_score(y_true, y_pred, zero_division=0))
        assert np.isclose(my_f1(y_true, y_pred), f1_score(y_true, y_pred, zero_division=0))


ipytest.run()

## Задание 2: Ловушка Accuracy

Сгенерируем сильно несбалансированный датасет (90% класс 0, 10% класс 1)
и покажем, почему accuracy бесполезна.

In [ ]:
# Generate imbalanced dataset — noisy, hard to separate, so baseline fails badly
X, y = make_classification(
    n_samples=2000, n_features=15, n_informative=5,
    n_redundant=3, n_clusters_per_class=2,
    weights=[0.90, 0.10], flip_y=0.05,
    class_sep=0.8, random_state=RANDOM_STATE
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: class 0 = {(y_train == 0).sum()}, class 1 = {(y_train == 1).sum()}")
print(f"Test:  class 0 = {(y_test == 0).sum()},  class 1 = {(y_test == 1).sum()}")


In [ ]:
#TODO: Train LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
# Store predictions in y_pred_lr
# ...

#TODO: Create dummy predictions - always predict class 0
# y_pred_dummy = ...

#TODO: Compute accuracy, precision, recall, F1 for both models using YOUR functions
# acc_lr, prec_lr, rec_lr, f1_lr = ...
# acc_dummy, prec_dummy, rec_dummy, f1_dummy = ...

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm_lr = sk_confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Logistic Regression')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

cm_dummy = sk_confusion_matrix(y_test, y_pred_dummy)
sns.heatmap(cm_dummy, annot=True, fmt='d', cmap='Reds', ax=axes[1])
axes[1].set_title('Dummy (always 0)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1']
lr_vals = [acc_lr, prec_lr, rec_lr, f1_lr]
dummy_vals = [acc_dummy, prec_dummy, rec_dummy, f1_dummy]
x = np.arange(len(metrics_names))
width = 0.35
axes[2].bar(x - width / 2, lr_vals, width, label='LR', color='steelblue')
axes[2].bar(x + width / 2, dummy_vals, width, label='Dummy', color='salmon')
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics_names)
axes[2].set_ylim(0, 1.1)
axes[2].set_title('Metrics Comparison')
axes[2].legend()
plt.tight_layout()
plt.show()

print(f"LR:    Accuracy={acc_lr:.3f}, Precision={prec_lr:.3f}, Recall={rec_lr:.3f}, F1={f1_lr:.3f}")
print(f"Dummy: Accuracy={acc_dummy:.3f}, Precision={prec_dummy:.3f}, Recall={rec_dummy:.3f}, F1={f1_dummy:.3f}")
print(f"\nDummy has high accuracy ({acc_dummy:.3f}) but Recall = {rec_dummy:.1f}!")

In [ ]:
assert acc_dummy > 0.85, "Dummy accuracy should be high due to imbalance"
assert rec_dummy == 0.0, "Dummy recall should be 0"
assert f1_lr > f1_dummy, "LR should have better F1"
assert rec_lr > rec_dummy, "LR should find at least some positives"
print("Tests passed!")

## Задание 3a: Random Oversampling

Дублируем объекты миноритарного класса случайным образом (с возвращением),
пока классы не сбалансируются.

In [ ]:
def random_oversample(X, y, random_state=None):
    '''Randomly oversample the minority class.
    Returns X_resampled, y_resampled with balanced classes.'''
    if random_state is not None:
        np.random.seed(random_state)

    #TODO:
    # 1. Find majority and minority classes
    # 2. Calculate how many samples to add
    # 3. Randomly select minority samples (with replacement) to duplicate
    # 4. Concatenate and shuffle
    pass

## Задание 3b: SMOTE (Synthetic Minority Oversampling Technique)

Для каждого объекта миноритарного класса:
1. Найти k ближайших соседей (среди миноритарного класса)
2. Выбрать случайного соседа
3. Создать синтетический объект:

$$x_{new} = x_i + \lambda \cdot (x_{nn} - x_i), \quad \lambda \sim U(0, 1)$$

In [ ]:
def smote_oversample(X, y, k_neighbors=5, random_state=None):
    '''SMOTE: generate synthetic minority class samples.
    Returns X_resampled, y_resampled with balanced classes.'''
    if random_state is not None:
        np.random.seed(random_state)

    #TODO:
    # 1. Separate minority class samples
    # 2. Calculate how many synthetic samples needed
    # 3. For each synthetic sample:
    #    a. Pick random minority sample
    #    b. Find k nearest neighbors (use np.linalg.norm)
    #    c. Pick random neighbor
    #    d. x_new = x + lambda * (x_neighbor - x), lambda ~ U(0,1)
    # 4. Concatenate and shuffle
    pass

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
X_ros, y_ros = random_oversample(X_train, y_train, random_state=RANDOM_STATE)
X_smote, y_smote = smote_oversample(X_train, y_train, random_state=RANDOM_STATE)

for ax, data, title in zip(
    axes,
    [(X_train, y_train), (X_ros, y_ros), (X_smote, y_smote)],
    ['Original', 'Random Oversampling', 'SMOTE'],
):
    Xd, yd = data
    ax.scatter(Xd[yd == 0, 0], Xd[yd == 0, 1], alpha=0.3, label='Class 0')
    ax.scatter(Xd[yd == 1, 0], Xd[yd == 1, 1], alpha=0.3, label='Class 1')
    ax.set_title(f'{title}\n(0: {(yd == 0).sum()}, 1: {(yd == 1).sum()})')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
class TestOversampling:
    def test_random_oversample_balanced(self):
        X = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])
        y = np.array([0, 0, 0, 1])
        X_r, y_r = random_oversample(X, y, random_state=42)
        assert sum(y_r == 0) == sum(y_r == 1)

    def test_smote_balanced(self):
        X, y = make_classification(n_samples=100, n_features=5,
                                   weights=[0.9, 0.1], random_state=42)
        X_s, y_s = smote_oversample(X, y, k_neighbors=5, random_state=42)
        assert sum(y_s == 0) == sum(y_s == 1)

    def test_smote_synthetic_in_range(self):
        # 3 minority (class 1) vs 6 majority (class 0) — actually imbalanced
        X_min = np.array([[0, 0], [2, 2], [4, 4]])
        y_min = np.array([1, 1, 1])
        X_maj = np.array([[10, 10]] * 6)
        y_maj = np.array([0] * 6)
        X = np.vstack([X_min, X_maj])
        y = np.concatenate([y_min, y_maj])
        X_s, y_s = smote_oversample(X, y, k_neighbors=2, random_state=42)
        minority = X_s[y_s == 1]
        assert minority.max() <= 4.0 + 1e-10
        assert minority.min() >= 0.0 - 1e-10
        assert sum(y_s == 0) == sum(y_s == 1)


ipytest.run()

## Задание 4: Random Undersampling

Удаляем объекты мажоритарного класса (случайно, без возвращения),
пока классы не сбалансируются.

In [ ]:
def random_undersample(X, y, random_state=None):
    '''Randomly undersample the majority class.
    Returns X_resampled, y_resampled with balanced classes.'''
    if random_state is not None:
        np.random.seed(random_state)

    #TODO:
    # 1. Find minority class count
    # 2. Randomly select that many majority samples (without replacement)
    # 3. Keep all minority samples
    # 4. Shuffle and return
    pass

In [ ]:
X_rus, y_rus = random_undersample(X_train, y_train, random_state=RANDOM_STATE)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, title in zip(axes, [(X_train, y_train), (X_rus, y_rus)],
                            ['Original', 'Random Undersampling']):
    Xd, yd = data
    ax.scatter(Xd[yd == 0, 0], Xd[yd == 0, 1], alpha=0.3, label='Class 0')
    ax.scatter(Xd[yd == 1, 0], Xd[yd == 1, 1], alpha=0.3, label='Class 1')
    ax.set_title(f'{title}\n(0: {(yd == 0).sum()}, 1: {(yd == 1).sum()})')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
class TestUndersampling:
    def test_undersample_balanced(self):
        X = np.array([[1], [2], [3], [4], [5], [6]])
        y = np.array([0, 0, 0, 0, 0, 1])
        X_r, y_r = random_undersample(X, y, random_state=42)
        assert sum(y_r == 0) == sum(y_r == 1)

    def test_preserves_minority(self):
        X, y = make_classification(n_samples=200, weights=[0.9, 0.1], random_state=42)
        X_r, y_r = random_undersample(X, y, random_state=42)
        assert sum(y_r == 1) == sum(y == 1)

    def test_smaller(self):
        X, y = make_classification(n_samples=200, weights=[0.9, 0.1], random_state=42)
        X_r, y_r = random_undersample(X, y, random_state=42)
        assert len(y_r) < len(y)


ipytest.run()

## Задание 5: Сравнение подходов

Сравните 5 подходов к работе с дисбалансом:
1. **Baseline** — без обработки
2. **class_weight='balanced'** — увеличение штрафа: $w_c = \frac{n}{2 \cdot n_c}$
3. **Random Oversampling + LR**
4. **SMOTE + LR**
5. **Random Undersampling + LR**

In [ ]:
# --- Approach 1: Baseline ---
#TODO: Train LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
# y_pred_baseline = ...

# --- Approach 2: class_weight ---
#TODO: Train LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, max_iter=1000)
# y_pred_weighted = ...

# --- Approach 3: Random Oversampling + LR ---
#TODO: Apply random_oversample, then train LR
# y_pred_ros = ...

# --- Approach 4: SMOTE + LR ---
#TODO: Apply smote_oversample, then train LR
# y_pred_smote = ...

# --- Approach 5: Undersampling + LR ---
#TODO: Apply random_undersample, then train LR
# y_pred_rus = ...

# Collect results (do not modify)
results = {}
approaches = {
    'Baseline': y_pred_baseline,
    'class_weight': y_pred_weighted,
    'Random OS': y_pred_ros,
    'SMOTE': y_pred_smote,
    'Random US': y_pred_rus,
}
for name, y_pred in approaches.items():
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
    }
results_df = pd.DataFrame(results).T
display(results_df)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
mnames = ['Accuracy', 'Precision', 'Recall', 'F1']
x = np.arange(len(mnames))
width = 0.15
colors = ['steelblue', 'darkorange', 'green', 'red', 'purple']
for i, (name, mets) in enumerate(results.items()):
    vals = [mets[m] for m in mnames]
    ax.bar(x + i * width, vals, width, label=name, color=colors[i], alpha=0.8)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(mnames)
ax.set_ylim(0, 1.1)
ax.set_title('Approaches Comparison')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
assert len(results) == 5, "Should have 5 approaches"
assert all('F1' in r for r in results.values()), "Each result must have F1"
assert results['class_weight']['Recall'] >= results['Baseline']['Recall'], \
    "class_weight should improve Recall"
print("Tests passed!")

## Задание 6: ROC и PR кривые

Постройте ROC и PR кривые для всех подходов из задания 5.
Для этого нужны **вероятности** предсказаний, а не классы.

In [ ]:
def get_proba(model, X_tr, y_tr, X_te):
    '''Train model and return probability of positive class.'''
    model.fit(X_tr, y_tr)
    return model.predict_proba(X_te)[:, 1]


#TODO: Get probabilities for each approach
# y_proba_baseline = get_proba(
#     LogisticRegression(random_state=RANDOM_STATE, max_iter=1000), X_train, y_train, X_test)
# y_proba_weighted = get_proba(
#     LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, max_iter=1000),
#     X_train, y_train, X_test)
# X_ros_p, y_ros_p = random_oversample(X_train, y_train, random_state=RANDOM_STATE)
# y_proba_ros = get_proba(
#     LogisticRegression(random_state=RANDOM_STATE, max_iter=1000), X_ros_p, y_ros_p, X_test)
# X_smote_p, y_smote_p = smote_oversample(X_train, y_train, random_state=RANDOM_STATE)
# y_proba_smote = get_proba(
#     LogisticRegression(random_state=RANDOM_STATE, max_iter=1000), X_smote_p, y_smote_p, X_test)
# X_rus_p, y_rus_p = random_undersample(X_train, y_train, random_state=RANDOM_STATE)
# y_proba_rus = get_proba(
#     LogisticRegression(random_state=RANDOM_STATE, max_iter=1000), X_rus_p, y_rus_p, X_test)

proba_dict = {
    'Baseline': y_proba_baseline,
    'class_weight': y_proba_weighted,
    'Random OS': y_proba_ros,
    'SMOTE': y_proba_smote,
    'Random US': y_proba_rus,
}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, y_proba in proba_dict.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curves'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

for name, y_proba in proba_dict.items():
    prec_vals, rec_vals, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    axes[1].plot(rec_vals, prec_vals, label=f'{name} (AP={ap:.3f})')
baseline_prec = y_test.sum() / len(y_test)
axes[1].axhline(baseline_prec, color='k', linestyle='--', label=f'Baseline ({baseline_prec:.2f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
auc_scores = {name: roc_auc_score(y_test, p) for name, p in proba_dict.items()}
assert len(auc_scores) == 5
assert all(v > 0.5 for v in auc_scores.values()), "All AUCs should be > 0.5"
print("AUC scores:", {k: f"{v:.3f}" for k, v in auc_scores.items()})
print("Tests passed!")

---

## Теория: Pipeline и предотвращение утечки данных

### Что такое Pipeline?

Pipeline — цепочка преобразований + финальная модель. Все шаги имеют общий интерфейс `fit`/`transform`/`predict`.

### Data Leakage (утечка данных)

**Неправильно:** fit scaler на ВСЕХ данных, потом split
**Правильно:** split, fit scaler на train, transform train и test

Pipeline автоматически гарантирует правильный порядок при cross-validation!

### sklearn Pipeline API

```python
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])
pipe.fit(X_train, y_train)
pipe.predict(X_test)
```

### ColumnTransformer

Для данных со смешанными типами (числа + категории):

```python
ct = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(), categorical_features)
])
```

## Задание 7: sklearn Pipeline

Создайте sklearn Pipeline для масштабирования и классификации.
Pipeline должен корректно масштабировать данные и обучать модель.

In [ ]:
# Generate data with different feature scales
X, y = make_classification(n_samples=500, n_features=5, n_informative=3,
                           n_redundant=1, random_state=RANDOM_STATE)
X[:, 0] *= 100
X[:, 1] *= 0.01
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE
)

#TODO: Create Pipeline with two steps:
# 1. 'scaler' -> StandardScaler()
# 2. 'classifier' -> LogisticRegression(random_state=RANDOM_STATE)
# pipe = Pipeline([...])

#TODO: Fit on X_train_p, y_train_p (unscaled!)
#TODO: Predict on X_test_p
# y_pred_pipe = ...

#TODO: Compute accuracy and F1
# acc_pipe = ...
# f1_pipe = ...

print(f"sklearn Pipeline: Accuracy={acc_pipe:.4f}, F1={f1_pipe:.4f}")


In [ ]:
assert hasattr(pipe, 'named_steps'), "pipe must be a Pipeline"
assert 'scaler' in pipe.named_steps
assert 'classifier' in pipe.named_steps
assert acc_pipe > 0.7, 'Pipeline accuracy should be > 0.7'
print("Tests passed!")

## Задание 8: ColumnTransformer для смешанных типов

Обработайте датасет с числовыми и категориальными признаками.

In [ ]:
# Generate mixed-type dataset
np.random.seed(RANDOM_STATE)
n = 500
df = pd.DataFrame({
    'age': np.random.randint(18, 70, n),
    'income': np.random.exponential(50000, n),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n),
    'city': np.random.choice(['Moscow', 'SPb', 'Kazan', 'Novosibirsk'], n),
    'experience_years': np.random.randint(0, 30, n),
})
y_mixed = ((df['income'] > 40000) & (df['education'].isin(['Master', 'PhD']))).astype(int)
flip_idx = np.random.choice(n, size=int(n * 0.1), replace=False)
y_mixed[flip_idx] = 1 - y_mixed[flip_idx]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    df, y_mixed, test_size=0.3, random_state=RANDOM_STATE
)
print(f"Class balance: {y_mixed.value_counts().to_dict()}")
print(f"Feature types:\n{df.dtypes}")

In [ ]:
numeric_features = ['age', 'income', 'experience_years']
categorical_features = ['education', 'city']

#TODO: Create ColumnTransformer:
# 1. StandardScaler for numeric_features
# 2. OneHotEncoder(drop='first', handle_unknown='ignore') for categorical_features
# preprocessor = ColumnTransformer(...)

#TODO: Create Pipeline: preprocessor -> LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
# pipe_mixed = Pipeline([...])

#TODO: Fit and predict
# y_pred_mixed = pipe_mixed.predict(X_test_m)

acc_mixed = accuracy_score(y_test_m, y_pred_mixed)
f1_mixed = f1_score(y_test_m, y_pred_mixed)
print(f"Mixed Pipeline: Accuracy={acc_mixed:.4f}, F1={f1_mixed:.4f}")

In [ ]:
assert hasattr(preprocessor, 'transformers'), "Must be ColumnTransformer"
assert len(preprocessor.transformers) == 2
assert acc_mixed > 0.5
assert len(pipe_mixed.predict(X_test_m.iloc[:3])) == 3
print("Tests passed!")

## Задание 9: Полный ML Pipeline

Объедините всё: SMOTE + Pipeline + cross_val_score.

**Примечание:** Стандартный sklearn Pipeline не умеет изменять `y` в промежуточных шагах,
поэтому oversampling применяется **до** пайплайна. В production используйте `imblearn.Pipeline`.

In [ ]:
#TODO: Apply SMOTE to training data
# X_train_smote, y_train_smote = smote_oversample(X_train, y_train, random_state=RANDOM_STATE)

#TODO: Create Pipeline: StandardScaler -> LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, max_iter=1000)
# full_pipe = Pipeline([...])

#TODO: Evaluate with cross_val_score + StratifiedKFold(5)
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# cv_scores_f1 = cross_val_score(full_pipe, X_train_smote, y_train_smote, cv=cv, scoring='f1')
# cv_scores_auc = cross_val_score(full_pipe, X_train_smote, y_train_smote, cv=cv, scoring='roc_auc')

print(f"CV F1:       {cv_scores_f1}")
print(f"CV F1 mean:  {cv_scores_f1.mean():.4f} (+/- {cv_scores_f1.std():.4f})")
print(f"CV AUC:      {cv_scores_auc}")
print(f"CV AUC mean: {cv_scores_auc.mean():.4f} (+/- {cv_scores_auc.std():.4f})")

#TODO: Fit on full training data and evaluate on test
# full_pipe.fit(X_train_smote, y_train_smote)
# y_pred_full = full_pipe.predict(X_test)
# final_f1 = f1_score(y_test, y_pred_full)
# final_auc = roc_auc_score(y_test, full_pipe.predict_proba(X_test)[:, 1])

print(f"\nTest F1:  {final_f1:.4f}")
print(f"Test AUC: {final_auc:.4f}")

In [ ]:
assert len(cv_scores_f1) == 5
assert cv_scores_f1.mean() > 0.3
assert final_f1 > 0.3
assert final_auc > 0.5
assert hasattr(full_pipe, 'named_steps')
print("Tests passed!")

## Final Tests

In [ ]:
class TestLab9:
    '''Comprehensive tests for the entire lab.'''

    def test_metrics_consistency(self):
        np.random.seed(RANDOM_STATE)
        y_true = np.random.randint(0, 2, 50)
        y_pred = np.random.randint(0, 2, 50)
        cm = my_confusion_matrix(y_true, y_pred)
        assert cm['TP'] + cm['FP'] + cm['FN'] + cm['TN'] == len(y_true)

    def test_oversample_undersample_roundtrip(self):
        X, y = make_classification(n_samples=200, weights=[0.9, 0.1], random_state=RANDOM_STATE)
        X_over, y_over = random_oversample(X, y, random_state=RANDOM_STATE)
        X_under, y_under = random_undersample(X_over, y_over, random_state=RANDOM_STATE)
        assert sum(y_under == 0) == sum(y_under == 1)

    def test_pipeline_no_data_leakage(self):
        X, y = make_classification(n_samples=200, n_features=5, random_state=RANDOM_STATE)
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE)
        p = Pipeline([('scaler', StandardScaler()),
                      ('clf', LogisticRegression(random_state=RANDOM_STATE))])
        p.fit(X_tr, y_tr)
        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_te_s = sc.transform(X_te)
        clf = LogisticRegression(random_state=RANDOM_STATE)
        clf.fit(X_tr_s, y_tr)
        assert np.allclose(p.predict(X_te), clf.predict(X_te_s))

    def test_smote_output_shape(self):
        X, y = make_classification(n_samples=100, n_features=5,
                                   weights=[0.8, 0.2], random_state=RANDOM_STATE)
        X_s, y_s = smote_oversample(X, y, k_neighbors=3, random_state=RANDOM_STATE)
        assert X_s.shape[1] == 5
        assert len(y_s) == X_s.shape[0]

    def test_column_transformer_pipeline(self):
        df_test = pd.DataFrame({
            'num1': [1.0, 2.0, 3.0, 4.0, 5.0],
            'cat1': ['a', 'b', 'a', 'b', 'a'],
        })
        y_t = np.array([0, 1, 0, 1, 0])
        ct = ColumnTransformer([
            ('num', StandardScaler(), ['num1']),
            ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), ['cat1']),
        ])
        p = Pipeline([('preprocessor', ct), ('clf', LogisticRegression())])
        p.fit(df_test, y_t)
        assert len(p.predict(df_test)) == 5


ipytest.run()

## Итоги

### Что вы узнали:

1. **Accuracy не подходит** для несбалансированных данных — используйте F1, ROC-AUC, PR-AUC
2. **Confusion matrix** — основа для понимания ошибок классификации
3. **Oversampling** (Random, SMOTE) — увеличивает представительство миноритарного класса
4. **Undersampling** — уменьшает мажоритарный класс (риск потери информации)
5. **class_weight='balanced'** — простейший способ учесть дисбаланс в sklearn
6. **Pipeline** — предотвращает утечку данных и упрощает код
7. **ColumnTransformer** — правильный способ обрабатывать смешанные типы признаков
8. **cross_val_score** с пайплайном — правильная оценка качества модели

### Ключевые формулы:
- $Precision = TP / (TP + FP)$
- $Recall = TP / (TP + FN)$
- $F_1 = 2 \cdot P \cdot R / (P + R)$
- SMOTE: $x_{new} = x_i + \lambda \cdot (x_{nn} - x_i)$